In [0]:
import requests
import pandas as pd
import time
from pyspark.sql.functions import upper, col, current_timestamp

url = "https://jsonplaceholder.typicode.com/users"
start_time = time.time()

print("Iniciando pipeline")

# Extração
response = requests.get(url)
response.raise_for_status()
data = response.json()

print(f"Registros recebidos: {len(data)}")

# Transformação (Pandas)
df_pandas = pd.DataFrame(data)
df_pandas["city"] = df_pandas["address"].apply(lambda x: x["city"])
df_pandas = df_pandas[["id", "name", "email", "city"]]

nulls = df_pandas.isnull().sum().sum()
print(f"Valores nulos: {nulls}")

# Conversão para Spark
df_spark = spark.createDataFrame(df_pandas)

# Transformação (Spark)
df_clean = df_spark.withColumn("name", upper(col("name"))) \
.withColumn("processing_time", current_timestamp())
display(df_clean)
print("Transformação concluída")

# Escrita Silver
df_clean.write.mode("overwrite").saveAsTable("silver_users")

# Query
df_clean.createOrReplaceTempView("users")
resultado = spark.sql("""
                      
SELECT city, COUNT(*) AS total_users
FROM users
GROUP BY city
ORDER BY total_users DESC """)

# Escrita Gold
resultado.write.mode("overwrite").saveAsTable("gold_users_by_city")

execution_time = round(time.time() - start_time, 2)
print(f"Tempo total: {execution_time}s")

display(resultado)
print("Finalizado")